# Auto-Docstring Tool (LLM-based)

Add PEP-257 docstrings to Python code using an LLM. **Full step-by-step documentation:** see [README.md](README.md) in this folder.

### Steps 1–5 overview

| Step | What it does |
|------|----------------|
| **1. Setup** | Imports, load env, check API key, create LLM client, define `SYSTEM_PROMPT`. |
| **2. Parse source** | `ast.parse(source)` and collect nodes: Module, ClassDef, FunctionDef, AsyncFunctionDef. |
| **3. Find undocumented targets** | Keep only nodes without a docstring; for each get name, kind, and code snippet. |
| **4. Process in reverse line order** | Sort targets by insert line descending so insertions don’t shift line numbers. |
| **5. Call LLM** | Build user prompt (style, kind, name, snippet), call API, get docstring text for each target. |
| **6. Insert docstrings** | For each target: get indent, format docstring block, insert at the correct line (reverse order so line numbers stay valid). |
| **7. Output** | Return the new source and count; optionally write to a file with `add_docstrings_to_file()`. |

## Setup

Run in order: env → client → SYSTEM_PROMPT. See README.md for details.

In [ ]:
# Imports
import os
from openai import OpenAI
import gradio as gr
from dotenv import load_dotenv
from IPython.display import Markdown, display
from docstring_utils import (
    parse_source,
    get_docstring_targets,
    find_undocumented_targets,
    sort_targets_for_insertion,
    get_insert_line,
    get_indent_for_node,
    insert_docstring_into_lines,
)

In [ ]:
load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if not api_key:
    print("No API key was found - please head over to the troubleshooting notebook in this folder to identify & fix!")
elif not api_key.startswith("sk-or-v1"):
    print("An API key was found, but it doesn't start sk-or-v1; please check you're using the right key - see troubleshooting notebook")
else:
    print("API key found and looks good so far!")

In [ ]:
openai = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=api_key)

openai_model = "openai/gpt-4o-mini"
gemini_model = "google/gemini-2.5-flash"

In [ ]:
SYSTEM_PROMPT = """
You are a Python documentation expert.
Your task is to generate a PEP 257–compliant docstring for the given Python code.

STRICT RULES:
- Output ONLY the inner docstring text. Do NOT include triple quotes.
- Do NOT output code, markdown, commentary, or explanation.
- Place the docstring as the first statement inside the corresponding function or class body.
- Do NOT generate standalone or top-level docstrings.
- Follow the requested style (Google, NumPy, or reStructuredText) exactly and consistently.
- Match parameters, return values, and exceptions EXACTLY as defined in the code.
- Do NOT invent parameters, exceptions, attributes, or behavior.
- For classes, document only the class purpose and attributes.
- Do NOT document method behavior inside the class docstring.
- Each method must have its own docstring.
- Do NOT include a Returns section for __init__ unless it explicitly returns a value.
- Use the exact function or class names in examples. Do NOT use placeholder names.
- Include sections only when applicable.
- Focus on intent, side effects, and non-obvious behavior.
- If behavior is unclear, document only what is directly inferable.
"""

### Step 2: (parse source): see README.md and docstring_utils.py

### Step 3: Find undocumented targets

**Goal:** From the list of docstring-capable nodes, keep only those that **don’t already have** a docstring, and for each one compute:
- **name** (e.g. function/class name, or `"<module>"` for the file),
- **kind** (module / class / function / async function),
- **snippet** (slice of source lines for that node so the LLM can see the code).

**How we detect “has docstring”:** In Python’s AST, a docstring is just the first statement in the body: an `Expr` node whose value is a `Constant` (string). So we check: if `node.body` is non-empty and that first statement is a string constant, we skip the node. Otherwise we mark it as needing a docstring.

**Snippet:** We slice `lines[node.lineno - 1 : end_lineno]` (with a cap, e.g. 30 lines) so the model sees the definition and enough context.

### Step 4 (reverse order): README.md, docstring_utils.sort_targets_for_insertion

(Sort so we **process the lowest in the file last** (i.e. by **descending** line number). That way, when we insert a docstring at line 5, we don’t change the line numbers of the function that starts at line 10. If we processed top-to-bottom, every insertion would shift the following lines and our stored line numbers would become wrong.

**Insertion line:** The docstring is inserted *before* the first statement of the node. For a function that starts at line 10, the first statement is usually line 11 (the first line of the body). So we use `node.body[0].lineno` (or for Module, the first line of the file) as the “insert at” line and sort by this value descending.

### Step 5: build_user_prompt + ask_llm_for_docstring (below)

User prompt builder for the LLM (style, kind, name, snippet).

In [ ]:
def build_user_prompt(style: str, kind: str, name: str, snippet: str) -> str:
    return f"""Write a {style}-style docstring for this Python {kind} '{name}':

{snippet}"""

### Step 6 (insert): docstring_utils.get_indent_for_node, format_docstring_block, insert_docstring_into_lines

(For each target we have the node, the docstring text from the LLM, and the current list of source lines. We compute the indent for that node, format the docstring as a block (triple quotes, correct indentation), and insert it at the correct line (before the first statement of the node). Because we process in reverse line order, each insertion does not change the line numbers of targets we haven’t processed yet.

## Pipeline (Steps 5 + 7): LLM call and add_docstrings_to_source

In [ ]:
def ask_llm_for_docstring(
    client, model: str, style: str, kind: str, name: str, snippet: str
) -> str:
    user_content = build_user_prompt(style, kind, name, snippet)
    resp = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_content},
        ],
        temperature=0.3,
    )
    return (resp.choices[0].message.content or "").strip()


def add_docstrings_to_source(
    source: str,
    style: str = "Google",
    model: str | None = None,
    client=None,
    max_snippet_lines: int = 30,
) -> tuple[str, int]:
    if client is None:
        client = openai
    if model is None:
        model = openai_model

    tree = parse_source(source)
    target_list = find_undocumented_targets(tree, source, max_snippet_lines)
    if not target_list:
        return source, 0

    sorted_list = sort_targets_for_insertion(target_list)
    lines = source.splitlines()
    added = 0

    for node, name, kind, snippet in sorted_list:
        doc_text = ask_llm_for_docstring(client, model, style, kind, name, snippet)
        if not doc_text:
            continue
        insert_line = get_insert_line(node) - 1  # 1-based -> 0-based
        indent = get_indent_for_node(lines, node)
        insert_docstring_into_lines(lines, insert_line, indent, doc_text)
        added += 1

    return "\n".join(lines), added

In [ ]:
def add_docstrings_to_file(
    filepath: str,
    style: str = "Google",
    model: str | None = None,
    client=None,
    out_path: str | None = None,
) -> tuple[str, int]:
   
    with open(filepath, encoding="utf-8") as f:
        source = f.read()
    new_source, count = add_docstrings_to_source(source, style=style, model=model, client=client)
    write_path = out_path or filepath
    with open(write_path, "w", encoding="utf-8") as f:
        f.write(new_source)
    return new_source, count


---
## Test Sample (Code)

**Option A – Paste code in the Gradio app below** and click "Add docstrings".

**Option B – Run in code:** Use a string with your Python code and call `add_docstrings_to_source(source, style="Google")`. Example with sample code:

In [ ]:
SAMPLE_CODE = '''
def fetch_user(user_id: int):
    conn = get_db_connection()
    row = conn.execute("SELECT * FROM users WHERE id = ?", (user_id,)).fetchone()
    return dict(row) if row else None

class DataLoader:
    def __init__(self, path: str):
        self.path = path
        self._cache = None

    def load(self):
        if self._cache is None:
            with open(self.path) as f:
                self._cache = json.load(f)
        return self._cache
'''

---
## User Interface

Paste or upload Python code, choose docstring style and model, then click **Add docstrings** to run the full pipeline. The output shows the code with generated docstrings and how many were added.

In [ ]:
def run_docstring_app(code: str, file, style: str, model: str) -> tuple[str, str]:
    """Gradio handler: run pipeline and return (output code, status message)."""
    if file is not None:
        try:
            path = file.name if hasattr(file, "name") else file
            if isinstance(path, list):
                path = path[0] if path else None
            if path:
                with open(path, encoding="utf-8") as f:
                    code = f.read()
        except Exception as e:
            return "", f"Error reading file: {e}"
    if not (code or "").strip():
        return "", "Paste Python code or upload a .py file."
    try:
        model_id = openai_model if model == "OpenAI (GPT-4o-mini)" else gemini_model
        new_source, count = add_docstrings_to_source(
            code, style=style, model=model_id, client=openai
        )
        return new_source, f"Done — added {count} docstring(s)."
    except SyntaxError as e:
        return code, f"Syntax error in source: {e}"
    except Exception as e:
        return code, f"Error: {e}"


with gr.Blocks(title="Auto-Docstring Tool", theme=gr.themes.Soft()) as demo:
    gr.Markdown("# Auto-Docstring Tool\nAdd PEP-257 docstrings to Python code using an LLM. Paste code or upload a `.py` file.")

    with gr.Row():
        with gr.Column(scale=1):
            style_drop = gr.Dropdown(
                choices=["Google", "NumPy", "reStructuredText"],
                value="Google",
                label="Docstring style",
            )
            model_drop = gr.Dropdown(
                choices=["OpenAI (GPT-4o-mini)", "Gemini (1.5 Flash)"],
                value="OpenAI (GPT-4o-mini)",
                label="Model",
            )
            file_in = gr.File(label="Upload .py file (optional)", file_types=[".py"])
            btn = gr.Button("Add docstrings", variant="primary")
            status = gr.Textbox(label="Status", interactive=False)

        with gr.Column(scale=2):
            code_in = gr.Code(
                language="python",
                label="Input code",
                value=SAMPLE_CODE.strip(),
                lines=20,
            )
            code_out = gr.Code(
                language="python",
                label="Output (with docstrings)",
                interactive=False,
                lines=20,
            )

    def load_file_into_code(f):
        if f is None:
            return ""
        path = f.name if hasattr(f, "name") else (f[0] if isinstance(f, list) and f else f)
        if not path:
            return ""
        try:
            with open(path, encoding="utf-8") as fp:
                return fp.read()
        except Exception:
            return ""

    file_in.change(fn=load_file_into_code, inputs=file_in, outputs=code_in)
    btn.click(
        fn=run_docstring_app,
        inputs=[code_in, file_in, style_drop, model_drop],
        outputs=[code_out, status],
    )

demo.launch(share=False)